# Smart Home Support Agent
A RAG-based support assistant for smart home devices using SentenceTransformers + FAISS.


In [11]:
# Install dependencies
!pip install -q sentence-transformers faiss-cpu gradio

In [12]:
# Imports
import json
import re
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

In [13]:
# Knowledge base  40 smart home issue-resolution pairs


guide_data_str = """
{
  "sections": {
    "Wi-Fi Connectivity Issues": ["Ensure your smart device is within Wi-Fi range.", "Restart your router and modem.", "Check if your Wi-Fi password has changed."],
    "Device Not Responding": ["Check if the device is plugged in and powered on.", "Try restarting the device.", "Verify the device app is updated to the latest version."],
    "Voice Command Problems": ["Speak clearly and close to the microphone.", "Check your device language settings.", "Ensure your internet connection is stable."],
    "App Not Connecting": ["Restart the app.", "Check for app updates.", "Reinstall the app if necessary."],
    "Light Not Turning On": ["Check if the bulb is screwed in properly.", "Verify the light switch is on.", "Ensure the bulb is compatible with your smart home system."],
    "Thermostat Not Changing Temperature": ["Check if the thermostat is in the correct mode (heating/cooling).", "Verify the set temperature.", "Ensure the HVAC system is powered on."],
    "Smart Lock Malfunction": ["Check battery levels.", "Recalibrate the lock.", "Ensure there are no obstructions to the bolt."],
    "Camera Offline": ["Check Wi-Fi connection.", "Restart the camera.", "Ensure power supply is stable."],
    "Speaker Not Playing Music": ["Check volume levels.", "Verify music service is linked and active.", "Restart the speaker."],
    "Sensor Not Detecting Activity": ["Check battery levels.", "Verify sensor placement.", "Ensure sensor is paired with the hub."],
    "Automation Not Triggering": ["Check automation rules for accuracy.", "Verify conditions are met.", "Ensure all devices involved are online."],
    "Remote Access Not Working": ["Check internet connectivity at home.", "Ensure mobile app has latest updates.", "Verify cloud services status."],
    "Firmware Update Failed": ["Ensure a stable internet connection during update.", "Restart device and try again.", "Contact manufacturer support if persistent."],
    "Device Lagging or Slow Response": ["Check network congestion.", "Restart device.", "Consider upgrading router or internet plan."],
    "Privacy Concerns with Smart Devices": ["Review device privacy settings.", "Understand data collection policies.", "Use strong, unique passwords."],
    "Compatibility Issues": ["Verify device compatibility with your smart home ecosystem.", "Check for firmware updates that might add compatibility.", "Consult the manufacturer compatibility list."],
    "Battery Draining Quickly": ["Adjust device settings to reduce power consumption.", "Check for background activity.", "Consider replacing the battery if old."],
    "Device Resetting Randomly": ["Check power supply stability.", "Look for firmware bugs and update if available.", "Contact support for hardware issues."],
    "Installation Difficulties": ["Follow the installation guide carefully.", "Watch online tutorials.", "Seek professional installation if needed."],
    "Notifications Not Working": ["Check app notification settings.", "Verify phone notification permissions.", "Ensure device is online and functioning."],
    "Account Login Problems": ["Reset your password.", "Clear app cache and data.", "Check for service outages."],
    "Device Pairing Issues": ["Ensure devices are in pairing mode.", "Bring devices closer to the hub.", "Restart both devices and try again."],
    "High Energy Consumption": ["Review device usage patterns.", "Adjust schedules and automation to conserve energy.", "Check for faulty devices."],
    "Data Storage Full": ["Delete old recordings and data.", "Upgrade to a larger storage plan.", "Adjust recording settings such as lower resolution."],
    "Interference with Other Devices": ["Change Wi-Fi channels.", "Relocate devices to reduce interference.", "Use wired connections where possible."],
    "Overheating Device": ["Ensure proper ventilation.", "Avoid direct sunlight.", "Reduce workload such as continuous streaming."],
    "Display Not Working": ["Restart device.", "Check brightness settings.", "Contact support for hardware failure."],
    "Sound Quality Issues": ["Check audio settings.", "Ensure speaker is placed correctly.", "Test with different audio sources."],
    "Button Not Responding": ["Clean button area.", "Restart device.", "Contact support for physical damage."],
    "Integration with Third-Party Services Failing": ["Re-link accounts.", "Check service statuses for outages.", "Ensure third-party app permissions are granted."],
    "Motion Detection Too Sensitive": {"Solution": "Adjust sensitivity settings in the device app.", "Detail": "Lower the sensitivity to reduce false alerts."},
    "Motion Detection Not Sensitive Enough": {"Solution": "Increase sensitivity settings in the device app.", "Detail": "Higher sensitivity will detect more subtle movements."},
    "Temperature Sensor Inaccurate": {"Solution": "Calibrate the sensor through the device app if available.", "Detail": "Ensure the sensor is not exposed to direct heat sources or drafts."},
    "Humidity Sensor Inaccurate": {"Solution": "Calibrate the sensor through the device app if available.", "Detail": "Verify the sensor is not near sources of moisture or extreme dryness."},
    "Water Leak Detector False Alarms": {"Solution": "Check sensor placement for potential false alarm triggers.", "Detail": "Ensure sensor is not in an area prone to condensation or minor spills."},
    "Water Leak Detector Not Alerting": {"Solution": "Test the sensor with a small amount of water to confirm functionality.", "Detail": "Check battery levels and connectivity to the smart home hub."},
    "Door/Window Sensor Not Registering": {"Solution": "Adjust the magnet and sensor alignment.", "Detail": "Ensure the gap between the magnet and sensor is within specified limits."},
    "Door/Window Sensor False Alerts": {"Solution": "Check for vibrations or loose mounting of the sensor.", "Detail": "Ensure the sensor and magnet are firmly attached to the door/window frame."},
    "Smoke Detector False Alarms": {"Solution": "Clean the smoke detector to remove dust and debris.", "Detail": "Ensure proper placement away from kitchens or high-humidity areas."},
    "Smoke Detector Not Alerting": {"Solution": "Test the smoke detector using its test button.", "Detail": "Check battery levels and ensure it is connected to the smart home system."}
  }
}
"""
guide = json.loads(guide_data_str)
print(f"Loaded {len(guide['sections'])} issue categories.")

Loaded 40 issue categories.


In [14]:
# Core helper functions

CONFIDENCE_THRESHOLD = 0.6  # Scores below this return 'no match found' instead of a weak result

def clean_text(text):
    """Normalize text: lowercase, remove punctuation, collapse whitespace."""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def extract_issues_from_guide(guide_data):
    """
    Extract (cleaned_issue, cleaned_resolution, original_resolution) triples.
    Preserves original resolution text for readable display while using
    cleaned text for embedding and retrieval.
    """
    issues = []
    for section_name, content in guide_data.get("sections", {}).items():
        issue_text = section_name.replace('_', ' ').title()

        if isinstance(content, list):
            resolution_original = ' '.join(map(str, content))
        elif isinstance(content, dict):
            resolution_original = ' '.join([f"{k}: {v}" for k, v in content.items()])
        else:
            resolution_original = str(content)

        cleaned_issue = clean_text(issue_text)
        cleaned_resolution = clean_text(resolution_original)

        if cleaned_issue and cleaned_resolution:
            issues.append((cleaned_issue, cleaned_resolution, resolution_original))
    return issues


def build_faiss_index(issues, model):
    """
    Build a FAISS IndexFlatIP (cosine similarity via normalized vectors).
    Returns index, issue texts, and original (display-ready) resolutions.
    """
    texts = [i[0] for i in issues]
    resolutions_display = [i[2] for i in issues]
    embeddings = model.encode(texts, convert_to_tensor=True, normalize_embeddings=True)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings.cpu().numpy())
    return index, texts, resolutions_display

In [15]:
# Tool 1 the retrieval_tool
# Semantic similarity search over the FAISS index
# This is the primary tool the agent calls to find matching issues

def retrieval_tool(user_query, index, texts, resolutions, model, top_k=3):
    """
    Tool 1: Semantic retrieval.
    Encodes the cleaned query, searches the FAISS index for the top_k
    most similar issues, and returns matches with confidence scores.
    """
    try:
        cleaned_query = clean_text(user_query)
        query_emb = model.encode([cleaned_query], convert_to_tensor=True, normalize_embeddings=True)
        D, I = index.search(query_emb.cpu().numpy(), top_k)
        results = []
        for idx, score in zip(I[0], D[0]):
            if idx < len(resolutions):
                results.append({
                    'similar_issue': texts[idx],
                    'resolution': resolutions[idx],
                    'score': float(score)
                })
        return results
    except Exception as e:
        print(f"[retrieval_tool] Error: {e}")
        return []

In [16]:
# Tool 2 the category_tool
# Pre-retrieval reasoning step: classifies the query into a device category
# This implements the ReAct 'Reason' step before the agent 'Acts' with retrieval

CATEGORY_KEYWORDS = {
    "lighting":    ["light", "bulb", "lamp", "dim", "flicker", "brightness"],
    "thermostat":  ["thermostat", "temperature", "heat", "cool", "hvac", "warm", "cold"],
    "lock":        ["lock", "door", "key", "bolt", "unlock", "entry"],
    "camera":      ["camera", "video", "recording", "motion", "offline", "stream"],
    "speaker":     ["speaker", "music", "sound", "audio", "alexa", "google home", "play"],
    "sensor":      ["sensor", "detect", "smoke", "leak", "water", "humidity", "window"],
    "network":     ["wifi", "wi-fi", "internet", "connect", "network", "router", "offline"],
    "app":         ["app", "phone", "notification", "login", "account", "update"],
    "general":     []
}

def category_tool(user_query):
    """
    Tool 2: Query categorization (ReAct reasoning step).
    Classifies the user query into a device category using keyword matching.
    Returns the category name and a brief reasoning note.
    This helps the agent contextualize what type of device problem is being described
    before delegating to the retrieval tool.
    """
    query_lower = user_query.lower()
    for category, keywords in CATEGORY_KEYWORDS.items():
        if any(kw in query_lower for kw in keywords):
            return {
                "category": category,
                "reasoning": f"Query contains keywords associated with '{category}' devices."
            }
    return {
        "category": "general",
        "reasoning": "No specific device category detected. Proceeding with general retrieval."
    }

In [17]:
# Load model and build index

model = None
faiss_index = None
issue_texts = []
issue_resolutions = []

if guide:
    try:
        issues = extract_issues_from_guide(guide)
        print(f"Extracted {len(issues)} issue-resolution pairs.")

        # all-MiniLM-L6-v2: stronger 384-dim embeddings vs the L3 variant
        model = SentenceTransformer('all-MiniLM-L6-v2')
        print("Model loaded: all-MiniLM-L6-v2")

        faiss_index, issue_texts, issue_resolutions = build_faiss_index(issues, model)
        print(f"FAISS index built with {faiss_index.ntotal} vectors.")

    except Exception as e:
        print(f"Setup error: {e}")
else:
    print("Guide data not loaded.")

Extracted 40 issue-resolution pairs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: all-MiniLM-L6-v2
FAISS index built with 40 vectors.


In [18]:
# ReAct agent loop
# Pattern: Observe to Reason (category_tool) to Act (retrieval_tool) to Respond

def run_agent(user_query):
    """
    ReAct agent loop:
      1. OBSERVE  — receive user query
      2. REASON   — call category_tool to classify device type
      3. ACT      — call retrieval_tool to find best matching issue
      4. RESPOND  — format and return answer with confidence score
    """
    if faiss_index is None or model is None:
        return "Agent not ready: index or model not loaded.", "", ""

    # Step 1: Observe
    trace = f"**Query:** {user_query}\n\n"

    # Step 2: Reason, use category_tool
    category_result = category_tool(user_query)
    trace += f"**Reasoning:** {category_result['reasoning']}  \n"
    trace += f"**Device category:** `{category_result['category']}`\n\n"

    # Step 3: Act,  use retrieval_tool
    results = retrieval_tool(user_query, faiss_index, issue_texts, issue_resolutions, model)

    # Step 4: Respond
    if not results or results[0]['score'] < CONFIDENCE_THRESHOLD:
        return (
            "No confident match found. Try rephrasing your issue (e.g. include the device type).",
            "",
            trace + "**Result:** Below confidence threshold — no answer returned."
        )

    best = results[0]
    confidence = f"{best['score'] * 100:.0f}%"
    suggested_fix = f"**Confidence: {confidence}**\n\n{best['resolution']}"

    related = "**Related issues:**\n"
    for r in results[1:]:
        related += f"- {r['similar_issue']} ({r['score'] * 100:.0f}%)\n"

    trace += f"**Top match:** {best['similar_issue']} ({confidence})"

    return suggested_fix, related, trace

In [19]:
# Cell 9: Gradio UI

import gradio as gr

if faiss_index is not None and model is not None:
    interface = gr.Interface(
        fn=run_agent,
        inputs=gr.Textbox(
            lines=2,
            placeholder="Describe your smart home issue, e.g. 'My thermostat won't change temperature'"
        ),
        outputs=[
            gr.Markdown(label="Suggested Fix"),
            gr.Markdown(label="Related Issues"),
            gr.Markdown(label="Agent Reasoning Trace")
        ],
        title="Smart Home Support Agent",
        description="Describe your smart home problem in plain English. The agent will classify it and find the best matching solution.",
        examples=[
            ["My Alexa won't respond to voice commands"],
            ["The smart lock battery keeps dying fast"],
            ["Camera keeps going offline"],
            ["Motion sensor is triggering too many false alerts"],
            ["I can't log into the app"],
        ]
    )
    interface.launch(debug=True)
else:
    print("Cannot launch: index or model not ready.")

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://49cc2b84535dc22c2c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://49cc2b84535dc22c2c.gradio.live
